# Tariikhna - Complete Chapter Pipeline

**End-to-end demonstration: The Hijra story**

This notebook takes the story manifest + passage schemas (already prepared) and:
1. Loads the story and all its passages
2. **Applies story-level character overrides** (ensures Abu Bakr looks the same across all 6 passages)
3. Builds enhanced image prompts with negative prompts for each panel (16 panels total)
4. Generates all images via FAL.ai
5. **Outputs a complete HTML chapter** that you can open in a browser

Estimated cost: ~$1 for 16 images at $0.025-0.04 each.

## 1. Setup

In [ ]:
!pip install fal-client Pillow requests -q


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
import time
import glob
import requests
from PIL import Image
from io import BytesIO
import base64

# ============================================
# SET YOUR FAL KEY (from https://fal.ai/dashboard/keys)
# ============================================
FAL_API_KEY  = "0de17aeb-1e1b-42a7-9c78-c9f0d2ee6b51:c30735f7b3792f63391ae672b69dc3d2"   # ← paste your fal.ai key
os.environ["FAL_KEY"] = FAL_API_KEY

import fal_client

# Output directory for the chapter
CHAPTER_DIR = 'hijra_chapter_output'
IMAGES_DIR = f'{CHAPTER_DIR}/images'
os.makedirs(IMAGES_DIR, exist_ok=True)

print('Setup complete!')

Setup complete!


## 2. Load Story Manifest and Passages

**IMPORTANT**: Place these files in the working directory before running:
- `00_story_manifest.json`
- `passage_01_plot.json`
- `passage_02_warning.json`
- `passage_03_preparation.json`
- `passage_04_escape.json`
- `passage_05_cave.json`
- `passage_06_arrival.json`

Or upload them to Colab using the file panel.

In [2]:
# Load the story manifest
with open('00_story_manifest.json') as f:
    story = json.load(f)

print(f'Story: {story["story_title"]}')
print(f'Era: {story["era"]}, Region: {story["region"]}')
print(f'Canonical characters: {len(story["canonical_characters"])}')
print(f'Passages: {len(story["passages"])}')

# Build a lookup for canonical characters
CANONICAL = {c['id']: c for c in story['canonical_characters']}
print(f'\nCharacter registry:')
for cid, char in CANONICAL.items():
    print(f'  {cid}: {char["name"]} [{char["depiction_rule"]}]')

# Load all passages in order
passages = []
passage_files = [
    'passage_01_plot.json',
    'passage_02_warning.json',
    'passage_03_preparation.json',
    'passage_04_escape.json',
    'passage_05_cave.json',
    'passage_06_arrival.json'
]

for i, fname in enumerate(passage_files):
    with open(fname) as f:
        p = json.load(f)
        p['_passage_id'] = story['passages'][i]['passage_id']
        p['_section_heading'] = story['passages'][i]['section_heading']
        passages.append(p)

total_panels = sum(len(p['output']['panels']) for p in passages)
print(f'\nLoaded {len(passages)} passages with {total_panels} total panels')

Story: The Hijra: The Great Journey to Madinah
Era: late_makkah, Region: makkah
Canonical characters: 6
Passages: 6

Character registry:
  prophet_muhammad: Prophet Muhammad [FROM_BEHIND]
  abu_bakr: Abu Bakr [FULL]
  asma: Asma [FULL]
  ali: Ali [FULL]
  abu_jahl: Abu Jahl [FULL]
  quraysh_leaders: The Quraysh Leaders [FULL]

Loaded 6 passages with 16 total panels


## 3. Apply Story-Level Character Overrides

This is critical for visual consistency. Each passage's character descriptions get **replaced** with the canonical version from the story manifest. This ensures Abu Bakr looks the same in passage 2 and passage 5.

In [3]:
def apply_canonical_overrides(passage, canonical):
    """Replace passage character descriptions with canonical ones."""
    new_chars = []
    for char in passage['output']['characters']:
        if char['id'] in canonical:
            # Use canonical version (preserves consistency)
            new_chars.append(canonical[char['id']])
        else:
            # Keep original (some characters may be passage-specific)
            new_chars.append(char)
    passage['output']['characters'] = new_chars
    return passage

for p in passages:
    apply_canonical_overrides(p, CANONICAL)

print('Canonical overrides applied to all passages.')
print('Characters are now consistent across the whole story.')

Canonical overrides applied to all passages.
Characters are now consistent across the whole story.


## 4. Build Enhanced Image Prompts (with Negative Prompts)

In [4]:
BASE_NEGATIVE = [
    'modern objects', 'electricity', 'cars', 'plastic', 'glass windows',
    'wristwatch', 'glasses', 'sneakers', 'photorealistic', 'photograph',
    'horror', 'gory', 'blood', 'violent', 'scary',
    'sexy', 'revealing clothing',
    'halo', 'divine rays', 'cross', 'idol'
]

def build_image_request(passage, panel, all_characters):
    """Build complete image gen request from a panel."""
    
    # Get character objects for those in this panel
    chars_in_panel = []
    char_lookup = {c['id']: c for c in all_characters}
    for cid in panel.get('characters_in_panel', []):
        if cid in char_lookup:
            chars_in_panel.append(char_lookup[cid])
    
    # Build positive prompt
    parts = [
        "Children's watercolor storybook illustration, soft warm colors, "
        "gentle rounded art style, child-friendly, hand-painted feel, picture book quality"
    ]
    
    parts.append(panel['image_prompt'])
    
    # Add explicit viewing instructions for restricted characters
    for char in chars_in_panel:
        rule = char.get('depiction_rule', 'FULL')
        name = char.get('name', 'character').split('(')[0].strip()
        
        if rule == 'FROM_BEHIND':
            parts.append(
                f"IMPORTANT: {name} must be shown entirely from behind, "
                f"face completely hidden, no facial features visible"
            )
        elif rule == 'SILHOUETTE':
            parts.append(f"{name} appears only as a dark silhouette without facial features")
        elif rule == 'NO_FACE':
            parts.append(f"{name}'s face must not be visible at all")
    
    parts.append("Historically accurate 7th century scene, no modern objects, no anachronisms")
    enhanced_prompt = ". ".join(parts)
    
    # Build negative prompt
    negative = list(BASE_NEGATIVE)
    for char in chars_in_panel:
        if char.get('depiction_rule') in ['NO_FACE', 'FROM_BEHIND', 'SILHOUETTE']:
            short_name = char['name'].split('(')[0].strip()
            negative.extend([
                f'face of {short_name}',
                f'frontal view of {short_name}',
                'visible facial features',
                'looking at viewer',
                'portrait view'
            ])
    
    return enhanced_prompt, ', '.join(negative)


# Preview one prompt
p = passages[0]
prompt, neg = build_image_request(p, p['output']['panels'][0], p['output']['characters'])
print(f'PROMPT ({len(prompt.split())} words):\n{prompt[:400]}...\n')
print(f'NEGATIVE ({len(neg.split())} words):\n{neg[:300]}...')

PROMPT (134 words):
Children's watercolor storybook illustration, soft warm colors, gentle rounded art style, child-friendly, hand-painted feel, picture book quality. Children's watercolor storybook illustration. Interior of an ornate 7th century Makkah meeting hall (Dar al-Nadwah). Stone walls hung with decorated rugs and tapestries. A circle of 6-7 wealthy older Arab men in their 40s-60s, sitting on cushioned bench...

NEGATIVE (25 words):
modern objects, electricity, cars, plastic, glass windows, wristwatch, glasses, sneakers, photorealistic, photograph, horror, gory, blood, violent, scary, sexy, revealing clothing, halo, divine rays, cross, idol...


## 5. Generate All Panel Images

In [7]:
MODEL_ENDPOINT = 'fal-ai/flux/dev'  # Change to flux-pro/v1.1, recraft-v3, etc.

def generate_image(prompt, negative_prompt, output_path):
    """Generate one image and save to disk."""
    try:
        result = fal_client.run(
            MODEL_ENDPOINT,
            arguments={
                'prompt': prompt,
                #'negative_prompt': negative_prompt,
                'image_size': 'landscape_4_3',
                'num_inference_steps': 28,
                'guidance_scale': 7.5,
                'num_images': 1,
            },
            #with_logs=False
        )
        
        url = None
        if 'images' in result and result['images']:
            url = result['images'][0]['url']
        elif 'image' in result:
            url = result['image']['url'] if isinstance(result['image'], dict) else result['image']
        
        if url:
            resp = requests.get(url, timeout=60)
            img = Image.open(BytesIO(resp.content))
            img.save(output_path)
            return True
    except Exception as e:
        print(f'    ERROR: {str(e)[:120]}')
    return False

# Generate all panels
total_panels = sum(len(p['output']['panels']) for p in passages)
current = 0
panel_metadata = []  # Track all panels for the HTML viewer

for p_idx, passage in enumerate(passages):
    section = passage['_section_heading']
    print(f'\n[{p_idx + 1}/{len(passages)}] {section}')
    
    for panel in passage['output']['panels']:
        current += 1
        panel_num = panel['panel_number']
        
        # Output filename
        img_name = f'p{p_idx + 1:02d}_panel{panel_num}.png'
        img_path = f'{IMAGES_DIR}/{img_name}'
        
        print(f'  [{current}/{total_panels}] Panel {panel_num}...', end=' ', flush=True)
        
        prompt, negative = build_image_request(passage, panel, passage['output']['characters'])
        
        success = generate_image(prompt, negative, img_path)
        
        panel_metadata.append({
            'passage_idx': p_idx,
            'panel_num': panel_num,
            'section_heading': section,
            'narrative_text': panel['narrative_text'],
            'image_filename': img_name,
            'success': success
        })
        
        if success:
            print('OK')
        else:
            print('FAILED')
        
        time.sleep(2)

successful = sum(1 for p in panel_metadata if p['success'])
print(f'\n\nGenerated {successful}/{total_panels} images successfully!')

# Save metadata
with open(f'{CHAPTER_DIR}/panels.json', 'w') as f:
    json.dump(panel_metadata, f, indent=2)


[1/6] The Dangerous Plot
  [1/16] Panel 1...     ERROR: HTTPSConnectionPool(host='v3b.fal.media', port=443): Max retries exceeded with url: /files/b/0a99293c/gL2k8hRRKiKOqj-9pf
FAILED
  [2/16] Panel 2... OK

[2/6] The Prophet Goes to Abu Bakr
  [3/16] Panel 1... OK
  [4/16] Panel 2...     ERROR: HTTPSConnectionPool(host='v3b.fal.media', port=443): Read timed out. (read timeout=60)
FAILED

[3/6] Asma Prepares the Provisions
  [5/16] Panel 1... OK
  [6/16] Panel 2... OK
  [7/16] Panel 3... OK

[4/6] The Escape from Makkah
  [8/16] Panel 1... OK
  [9/16] Panel 2... OK
  [10/16] Panel 3... OK

[5/6] Hiding in the Cave of Thawr
  [11/16] Panel 1... OK
  [12/16] Panel 2... OK
  [13/16] Panel 3...     ERROR: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', Non
FAILED

[6/6] Arrival in Madinah
  [14/16] Panel 1...     ERROR: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the

## 6. Build the HTML Chapter Viewer

In [ ]:
# Group panels by passage for the HTML
by_passage = {}
for panel in panel_metadata:
    pidx = panel['passage_idx']
    if pidx not in by_passage:
        by_passage[pidx] = {
            'section_heading': panel['section_heading'],
            'panels': []
        }
    by_passage[pidx]['panels'].append(panel)

# Build HTML
html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>{story["story_title"]} | Tariikhna</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Cormorant+Garamond:ital,wght@0,400;0,500;0,600;0,700;1,400&family=Amiri:ital,wght@0,400;0,700;1,400&display=swap" rel="stylesheet">
<style>
  :root {{
    --paper: #f5ede0;
    --ink: #2d1810;
    --gold: #c4914a;
    --gold-deep: #8b6332;
    --rust: #8b3a1f;
    --shadow: rgba(45, 24, 16, 0.15);
  }}
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{
    font-family: 'Cormorant Garamond', serif;
    background: var(--paper);
    color: var(--ink);
    background-image:
      radial-gradient(ellipse at top, rgba(196, 145, 74, 0.08), transparent 70%),
      radial-gradient(ellipse at bottom, rgba(139, 58, 31, 0.06), transparent 70%);
    background-attachment: fixed;
    line-height: 1.7;
    min-height: 100vh;
  }}
  .ornament-top {{
    text-align: center;
    padding: 3rem 1rem 1rem;
    color: var(--gold);
    font-size: 1.5rem;
    letter-spacing: 1rem;
  }}
  .container {{
    max-width: 850px;
    margin: 0 auto;
    padding: 2rem 2rem 6rem;
  }}
  .cover {{
    text-align: center;
    padding: 4rem 0 5rem;
    border-bottom: 2px solid var(--gold);
    margin-bottom: 4rem;
    position: relative;
  }}
  .cover::before, .cover::after {{
    content: '';
    position: absolute;
    bottom: -1px;
    width: 100px;
    height: 2px;
    background: var(--gold-deep);
  }}
  .cover::before {{ left: calc(50% - 130px); }}
  .cover::after {{ right: calc(50% - 130px); }}
  .cover-eyebrow {{
    font-style: italic;
    color: var(--gold-deep);
    letter-spacing: 0.3em;
    font-size: 0.95rem;
    margin-bottom: 1.5rem;
    text-transform: uppercase;
  }}
  .cover h1 {{
    font-size: clamp(2.5rem, 6vw, 4.5rem);
    font-weight: 600;
    line-height: 1.1;
    color: var(--ink);
    margin-bottom: 2rem;
  }}
  .cover-summary {{
    font-size: 1.25rem;
    font-style: italic;
    color: var(--gold-deep);
    max-width: 600px;
    margin: 0 auto 2rem;
    line-height: 1.6;
  }}
  .cover-meta {{
    font-size: 0.9rem;
    color: var(--rust);
    letter-spacing: 0.1em;
  }}
  .cover-meta span {{ margin: 0 0.75rem; }}
  .introduction {{
    font-size: 1.2rem;
    font-style: italic;
    text-align: center;
    margin: 3rem auto 5rem;
    max-width: 700px;
    color: var(--ink);
    padding: 2rem 3rem;
    border-top: 1px solid var(--gold);
    border-bottom: 1px solid var(--gold);
    position: relative;
  }}
  .introduction::first-letter {{
    font-size: 3rem;
    font-weight: 600;
    color: var(--rust);
    float: left;
    margin: 0.2rem 0.5rem 0 0;
    font-style: normal;
  }}
  .section {{
    margin-bottom: 4rem;
  }}
  .section-header {{
    text-align: center;
    margin-bottom: 2.5rem;
  }}
  .section-number {{
    color: var(--gold);
    font-style: italic;
    font-size: 1.1rem;
    letter-spacing: 0.3em;
    margin-bottom: 0.5rem;
    text-transform: uppercase;
  }}
  .section-title {{
    font-size: 2.2rem;
    font-weight: 500;
    color: var(--ink);
  }}
  .section-divider {{
    text-align: center;
    color: var(--gold);
    font-size: 1.5rem;
    letter-spacing: 1rem;
    margin: 1rem 0 0;
  }}
  .panel {{
    margin-bottom: 3rem;
    background: rgba(255, 252, 245, 0.6);
    border-radius: 4px;
    overflow: hidden;
    box-shadow: 0 4px 24px var(--shadow);
  }}
  .panel img {{
    width: 100%;
    display: block;
    border-bottom: 8px solid var(--gold-deep);
  }}
  .panel-text {{
    padding: 2rem 2.5rem;
    font-size: 1.2rem;
    line-height: 1.8;
    color: var(--ink);
  }}
  .panel-text::first-letter {{
    font-size: 2.5rem;
    font-weight: 600;
    color: var(--rust);
    float: left;
    margin: 0.25rem 0.5rem 0 0;
    line-height: 1;
  }}
  .panel-image-missing {{
    background: rgba(139, 99, 50, 0.1);
    padding: 4rem;
    text-align: center;
    color: var(--gold-deep);
    font-style: italic;
    border-bottom: 8px solid var(--gold-deep);
  }}
  .conclusion {{
    font-size: 1.2rem;
    font-style: italic;
    text-align: center;
    margin: 5rem auto 3rem;
    max-width: 700px;
    color: var(--ink);
    padding: 3rem 2rem;
    line-height: 1.7;
  }}
  .conclusion::before {{
    content: '✦';
    display: block;
    color: var(--gold);
    font-size: 2rem;
    margin-bottom: 1.5rem;
  }}
  .sources {{
    margin-top: 4rem;
    padding: 2rem;
    border-top: 1px solid var(--gold);
    text-align: center;
    font-size: 0.9rem;
    color: var(--gold-deep);
    font-style: italic;
  }}
  .sources strong {{
    display: block;
    text-transform: uppercase;
    letter-spacing: 0.2em;
    font-style: normal;
    margin-bottom: 0.75rem;
    color: var(--rust);
  }}
  @media (max-width: 600px) {{
    .container {{ padding: 1rem 1rem 3rem; }}
    .panel-text {{ padding: 1.5rem 1.5rem; font-size: 1.1rem; }}
    .cover {{ padding: 2rem 0 3rem; }}
  }}
</style>
</head>
<body>
<div class="ornament-top">✦ ✦ ✦</div>
<div class="container">
  <div class="cover">
    <div class="cover-eyebrow">From the Sira of the Prophet</div>
    <h1>{story["story_title"]}</h1>
    <p class="cover-summary">{story["story_summary"]}</p>
    <div class="cover-meta">
      <span>Ages {story["age_group"]}</span>·<span>{story["estimated_reading_time_minutes"]} min read</span>
    </div>
  </div>
  <div class="introduction">{story["story_introduction"]}</div>
'''

for p_idx in sorted(by_passage.keys()):
    section = by_passage[p_idx]
    html += f'''
  <div class="section">
    <div class="section-header">
      <div class="section-number">Part {p_idx + 1}</div>
      <div class="section-title">{section["section_heading"]}</div>
      <div class="section-divider">✦ ✦ ✦</div>
    </div>
'''
    for panel in section['panels']:
        if panel['success']:
            html += f'''
    <div class="panel">
      <img src="images/{panel["image_filename"]}" alt="Panel {panel["panel_num"]}">
      <div class="panel-text">{panel["narrative_text"]}</div>
    </div>
'''
        else:
            html += f'''
    <div class="panel">
      <div class="panel-image-missing">[ Image generation failed for this panel ]</div>
      <div class="panel-text">{panel["narrative_text"]}</div>
    </div>
'''
    html += '  </div>\n'

# Conclusion + sources
sources_list = ' · '.join(story['scholarly_sources'])
html += f'''
  <div class="conclusion">{story["story_conclusion"]}</div>
  <div class="sources">
    <strong>Scholarly Sources</strong>
    {sources_list}
  </div>
</div>
<div class="ornament-top">✦ ✦ ✦</div>
</body>
</html>'''

# Save
with open(f'{CHAPTER_DIR}/index.html', 'w', encoding='utf-8') as f:
    f.write(html)

print(f'Chapter HTML created at: {CHAPTER_DIR}/index.html')
print(f'Open this file in a browser to view the complete chapter.')
print(f'Total panels included: {sum(len(s["panels"]) for s in by_passage.values())}')

## 7. (Optional) Display Inline in Notebook

In [ ]:
from IPython.display import IFrame, HTML, display
import os

# Display the chapter inline (works in Colab too)
with open(f'{CHAPTER_DIR}/index.html') as f:
    chapter_html = f.read()

# For inline display, we need to inline the images as base64
# (only do this if you want to see it in the notebook)
import base64

modified_html = chapter_html
for panel in panel_metadata:
    if panel['success']:
        img_path = f'{IMAGES_DIR}/{panel["image_filename"]}'
        if os.path.exists(img_path):
            with open(img_path, 'rb') as f:
                b64 = base64.b64encode(f.read()).decode()
            modified_html = modified_html.replace(
                f'images/{panel["image_filename"]}',
                f'data:image/png;base64,{b64}'
            )

# Save the standalone version (everything inline)
with open(f'{CHAPTER_DIR}/standalone.html', 'w', encoding='utf-8') as f:
    f.write(modified_html)

print(f'Standalone version: {CHAPTER_DIR}/standalone.html')
print('(All images embedded - single shareable file)')

# Display
display(HTML(modified_html))

## Summary

You now have:
- `hijra_chapter_output/index.html` - the chapter viewer (loads images from folder)
- `hijra_chapter_output/standalone.html` - single-file version (images embedded)
- `hijra_chapter_output/images/` - all 16 generated panel images
- `hijra_chapter_output/panels.json` - metadata for every panel

**The complete pipeline demonstrates:**
1. Editorial story manifest (curated by humans)
2. v3 schemas (one per passage, multi-panel where needed)
3. Story-level character consistency overrides
4. Enhanced prompts with negative prompts and depiction rules
5. Image generation via FAL.ai
6. Final reader-ready HTML output